In [0]:
# Importaciones
import pandas as pd
from pyspark.sql.functions import lit

In [0]:
# Descargar y combinar los dos archivos CSV
URL_RED   = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
URL_WHITE = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"

df_red_pd   = pd.read_csv(URL_RED,   sep=";")
df_white_pd = pd.read_csv(URL_WHITE, sep=";")

# Agregar columna de tipo de vino
df_red_pd["wine_type"]   = "red"
df_white_pd["wine_type"] = "white"

# Combinar en un solo DataFrame de pandas
df_pd = pd.concat([df_red_pd, df_white_pd], ignore_index=True)

print(f"Filas totales: {len(df_pd)}")
print(f"Columnas: {list(df_pd.columns)}")
print(df_pd.head())

In [0]:
# Convertir a Spark DataFrame y guardar como Delta
df_spark = spark.createDataFrame(df_pd)

# Renombrar columnas: los espacios causan problemas en SQL
new_cols = [c.replace(" ", "_") for c in df_spark.columns]
df_spark = df_spark.toDF(*new_cols)

df_spark.printSchema()

# Guardar como tabla Delta permanente (disponible para todos los notebooks)
df_spark.write.format("delta").mode("overwrite").saveAsTable("wine_quality_raw")

print("✅ Tabla 'wine_quality_raw' creada exitosamente.")

In [0]:
# Verificar que la tabla quedó bien
display(spark.sql("SELECT * FROM wine_quality_raw LIMIT 10"))